In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import pandas as pd

In [2]:
BASE_URL = "https://books.toscrape.com/"
OUTPUT_FILE = "books.csv"
DELAY = 0.5
MAX_PAGES = 50

In [3]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 "})

In [4]:
def get_soup(url):
    resp = session.get(url)
    resp.raise_for_status()
    resp.encoding = "utf-8"   # fixes the pound sign showing as garbage
    return BeautifulSoup(resp.text, "html.parser")

def scrape_book_page(url):
    soup = get_soup(url)
    if soup is None:
        return {"description": "", "category": "", "upc": "", "stock": ""}

    desc_tag = soup.select_one("#product_description + p")
    description = desc_tag.text.strip() if desc_tag else ""

    crumbs = soup.select("ul.breadcrumb li a")
    category = crumbs[2].text.strip() if len(crumbs) > 2 else ""

    info = {row.th.text.strip(): row.td.text.strip()
            for row in soup.select("table.table-striped tr")}

    return {
        "description": description,
        "category": category,
        "upc": info.get("UPC", ""),
        "stock": info.get("Availability", ""),
    }


def scrape_all_books():
    rows = []
    page_url = BASE_URL
    page_no = 1

    while page_url and page_no <= MAX_PAGES:
        print(f"Scraping page {page_no}: {page_url}")
        soup = get_soup(page_url)
        if soup is None:
            break

        for book in soup.select("article.product_pod"):
            link = book.select_one("h3 a")
            book_url = urljoin(page_url, link["href"])
            details = scrape_book_page(book_url)
            time.sleep(DELAY)

            rows.append({
                "title": link["title"],
                "category": details["category"],
                "price": book.select_one("p.price_color").text.strip(),
                "rating": book.select_one("p.star-rating")["class"][1],
                "availability": details["stock"],
                "upc": details["upc"],
                "description": details["description"],
                "url": book_url,
            })

        next_btn = soup.select_one("li.next a")
        page_url = urljoin(page_url, next_btn["href"]) if next_btn else None
        page_no += 1
        time.sleep(DELAY)

    return rows


rows = scrape_all_books()
print(f"\nScraped {len(rows)} books")

Scraping page 1: https://books.toscrape.com/
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Scraping page 6: https://books.toscrape.com/catalogue/page-6.html
Scraping page 7: https://books.toscrape.com/catalogue/page-7.html
Scraping page 8: https://books.toscrape.com/catalogue/page-8.html
Scraping page 9: https://books.toscrape.com/catalogue/page-9.html
Scraping page 10: https://books.toscrape.com/catalogue/page-10.html
Scraping page 11: https://books.toscrape.com/catalogue/page-11.html
Scraping page 12: https://books.toscrape.com/catalogue/page-12.html
Scraping page 13: https://books.toscrape.com/catalogue/page-13.html
Scraping page 14: https://books.toscrape.com/catalogue/page-14.html
Scraping page 15: https://books.toscrape.com/catalogue/page-15.html
Scraping page 16: h

In [5]:
df = pd.DataFrame(rows)
df.head()

,title,category,price,rating,availability,upc,description,url
0,A Light in the Attic,Poetry,£51.77,Three,In stock (22 available),a897fe39b1053632,It's hard to imagine a world without A Light i...,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Historical Fiction,£53.74,One,In stock (20 available),90fa61229261140a,"""Erotic and absorbing...Written with starling ...",https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Fiction,£50.10,One,In stock (20 available),6957f44c3847a760,"Dans une France assez proche de la nôtre, un h...",https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Mystery,£47.82,Four,In stock (20 available),e00eb4fd7b871a48,"WICKED above her hipbone, GIRL across her hear...",https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,History,£54.23,Five,In stock (20 available),4165285e1663650f,From a renowned historian comes a groundbreaki...,https://books.toscrape.com/catalogue/sapiens-a...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         1000 non-null   object
 1   category      1000 non-null   object
 2   price         1000 non-null   object
 3   rating        1000 non-null   object
 4   availability  1000 non-null   object
 5   upc           1000 non-null   object
 6   description   1000 non-null   object
 7   url           1000 non-null   object
dtypes: object(8)
memory usage: 62.6+ KB


In [7]:
#Converting to float and removed symbol
df["price"] = df["price"].str.replace(r"[^\d.]", "", regex=True).astype(float)

#Convert to intger
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating"].map(rating_map).astype(int)

# Convert to integer value
df["availability"] = (df["availability"].str.extract(r"(\d+)")[0].fillna(0).astype(int))

In [8]:
#  combined Column 
df["text"] = (
    "Title: "
    + df["title"]
    + "\nCategory: "
    + df["category"]
    + "\nPrice: £"
    + df["price"].astype(str)
    + "\nRating: "
    + df["rating"].astype(str)
    + "/5"
    + "\nAvailability: "
    + df["availability"].astype(str)
    + " copies available"
    + "\nDescription: "
    + df["description"]
)

df.to_csv("books.csv", index=False, encoding="utf-8")